# Project 4: Spotify — Pass 07a

## Problem Definition


#### Business Problem

This project aims to predict the popularity of unreleased songs shortly following release, based on song data from Spotify's Top 200 Weekly (Global) charts from 2020 and 2021. Companies like Spotify may wish to understand and predict what types of tracks become hits and go viral, as these tracks can be promoted better and charge more for advertising.

#### Data Science Problem

Song popularity will be predicted using one of three different tree-based regression models: decision-tree, random forest, and xgboost.

Popularity is a continuous, numeric floating-point value. As such, this is a supervised regression problem, where the target of the prediction is the upcoming popularity of a prospective track release.

(Note: For passes 1-3, transactions data from Project 1 will be used. Similarly, for passes 4-6, housing data from Project 2 will be used. Passes 7-10 will utilize the spotify data particular to the above problem statement, with pass 10 having the most comprehensive use.)

In [7]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets, metrics
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

import requests
from google.colab import userdata
import os

## Data Collection/Sources


### Read parquet from Hugging Face


In [1]:
# Construct the URL
url = f"https://ddc-datascience.s3.amazonaws.com/Projects/Project.4-Spotify/Data/Spotify.csv"
url


'https://ddc-datascience.s3.amazonaws.com/Projects/Project.4-Spotify/Data/Spotify.csv'

IDA Triage in bash

In [2]:
!curl -s -I {url}

HTTP/1.1 200 OK
x-amz-id-2: bNV/XseFCS239RmqA/SRC/sVALUFTCxdKUfo3fhwUJ6LOOP5vo9z1TWwVzKN7WPsALql5UrmouHZQcXYZ1Nh8wRtab2QJryX
x-amz-request-id: KKTEC87ZTKZQFPE5
Server: AmazonS3
Date: Tue, 24 Nov 2026 10:14:47 GMT
Last-Modified: Wed, 04 Oct 2023 17:23:56 GMT
ETag: "65b9875b11e0d7ea03ee2af024f45e99"
x-amz-server-side-encryption: AES256
Accept-Ranges: bytes
Content-Length: 738124
Content-Type: text/csv



In [3]:
!curl -s -O {url}

In [4]:
ls -la

total 740
drwxr-xr-x 1 root root   4096 Nov 24 10:15 ./
drwxr-xr-x 1 root root   4096 Nov 24 10:06 ../
drwxr-xr-x 4 root root   4096 Nov 18 15:58 .config/
drwxr-xr-x 1 root root   4096 Nov 18 15:58 sample_data/
-rw-r--r-- 1 root root 738124 Nov 24 10:15 Spotify.csv


In [5]:
!head -1 Spotify.csv | tr , '\n' | cat -n

     1	Index
     2	Highest Charting Position
     3	Number of Times Charted
     4	Week of Highest Charting
     5	Song Name
     6	Streams
     7	Artist
     8	Artist Followers
     9	Song ID
    10	Genre
    11	Release Date
    12	Weeks Charted
    13	Popularity
    14	Danceability
    15	Energy
    16	Loudness
    17	Speechiness
    18	Acousticness
    19	Liveness
    20	Tempo
    21	Duration (ms)
    22	Valence
    23	Chord


Import data into a Pandas dataframe

In [8]:
df = pd.read_csv( url )
df.shape

(1556, 23)

In [9]:
df.iloc[:,:5].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1556 entries, 0 to 1555
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Index                      1556 non-null   int64 
 1   Highest Charting Position  1556 non-null   int64 
 2   Number of Times Charted    1556 non-null   int64 
 3   Week of Highest Charting   1556 non-null   object
 4   Song Name                  1556 non-null   object
dtypes: int64(3), object(2)
memory usage: 60.9+ KB


## Data Cleaning


Basic Cleaning/Inspection

In [10]:
df.head()

,Index,Highest Charting Position,Number of Times Charted,Week of Highest Charting,Song Name,Streams,Artist,Artist Followers,Song ID,Genre,...,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence,Chord
0,1,1,8,2021-07-23--2021-07-30,Beggin',"48,633,449",Måneskin,3377762,3Wrjm47oTz2sjIgck11l5e,"['indie rock italiano', 'italian pop']",...,0.714,0.8,-4.808,0.0504,0.127,0.359,134.002,211560,0.589,B
1,2,2,3,2021-07-23--2021-07-30,STAY (with Justin Bieber),"47,248,719",The Kid LAROI,2230022,5HCyWlXZPP0y6Gqq8TgA20,['australian hip hop'],...,0.591,0.764,-5.484,0.0483,0.0383,0.103,169.928,141806,0.478,C#/Db
2,3,1,11,2021-06-25--2021-07-02,good 4 u,"40,162,559",Olivia Rodrigo,6266514,4ZtFanR9U6ndgddUvNcjcG,['pop'],...,0.563,0.664,-5.044,0.154,0.335,0.0849,166.928,178147,0.688,A
3,4,3,5,2021-07-02--2021-07-09,Bad Habits,"37,799,456",Ed Sheeran,83293380,6PQ88X9TkUIAUIZJHW2upE,"['pop', 'uk pop']",...,0.808,0.897,-3.712,0.0348,0.0469,0.364,126.026,231041,0.591,B
4,5,5,1,2021-07-23--2021-07-30,INDUSTRY BABY (feat. Jack Harlow),"33,948,454",Lil Nas X,5473565,27NovPIUIRrOZoCHxABJwK,"['lgbtq+ hip hop', 'pop rap']",...,0.736,0.704,-7.409,0.0615,0.0203,0.0501,149.995,212000,0.894,D#/Eb


In [11]:
df.tail()

,Index,Highest Charting Position,Number of Times Charted,Week of Highest Charting,Song Name,Streams,Artist,Artist Followers,Song ID,Genre,...,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence,Chord
1551,1552,195,1,2019-12-27--2020-01-03,New Rules,"4,630,675",Dua Lipa,27167675,2ekn2ttSfGqwhhate0LSR0,"['dance pop', 'pop', 'uk pop']",...,0.762,0.7,-6.021,0.0694,0.00261,0.153,116.073,209320,0.608,A
1552,1553,196,1,2019-12-27--2020-01-03,Cheirosa - Ao Vivo,"4,623,030",Jorge & Mateus,15019109,2PWjKmjyTZeDpmOUa3a5da,"['sertanejo', 'sertanejo universitario']",...,0.528,0.87,-3.123,0.0851,0.24,0.333,152.37,181930,0.714,B
1553,1554,197,1,2019-12-27--2020-01-03,Havana (feat. Young Thug),"4,620,876",Camila Cabello,22698747,1rfofaqEpACxVEHIZBJe6W,"['dance pop', 'electropop', 'pop', 'post-teen ...",...,0.765,0.523,-4.333,0.03,0.184,0.132,104.988,217307,0.394,D
1554,1555,198,1,2019-12-27--2020-01-03,Surtada - Remix Brega Funk,"4,607,385","Dadá Boladão, Tati Zaqui, OIK",208630,5F8ffc8KWKNawllr5WsW0r,"['brega funk', 'funk carioca']",...,0.832,0.55,-7.026,0.0587,0.249,0.182,154.064,152784,0.881,F
1555,1556,199,1,2019-12-27--2020-01-03,Lover (Remix) [feat. Shawn Mendes],"4,595,450",Taylor Swift,42227614,3i9UVldZOE0aD0JnyfAZZ0,"['pop', 'post-teen pop']",...,0.448,0.603,-7.176,0.064,0.433,0.0862,205.272,221307,0.422,G


In [12]:
df.shape

(1556, 23)

In [13]:
df.size

35788

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1556 entries, 0 to 1555
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Index                      1556 non-null   int64 
 1   Highest Charting Position  1556 non-null   int64 
 2   Number of Times Charted    1556 non-null   int64 
 3   Week of Highest Charting   1556 non-null   object
 4   Song Name                  1556 non-null   object
 5   Streams                    1556 non-null   object
 6   Artist                     1556 non-null   object
 7   Artist Followers           1556 non-null   object
 8   Song ID                    1556 non-null   object
 9   Genre                      1556 non-null   object
 10  Release Date               1556 non-null   object
 11  Weeks Charted              1556 non-null   object
 12  Popularity                 1556 non-null   object
 13  Danceability               1556 non-null   object
 14  Energy  

In [14]:
df.describe().transpose().head()

,count,mean,std,min,25%,50%,75%,max
Index,1556.0,778.500000,449.322824,1.0,389.75,778.5,1167.25,1556.0
Highest Charting Position,1556.0,87.744216,58.147225,1.0,37.00,80.0,137.00,200.0
Number of Times Charted,1556.0,10.668380,16.360546,1.0,1.00,4.0,12.00,142.0


In [15]:
df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
Index,1556.0,778.500000,449.322824,1.0,389.75,778.5,1167.25,1556.0
Highest Charting Position,1556.0,87.744216,58.147225,1.0,37.00,80.0,137.00,200.0
Number of Times Charted,1556.0,10.668380,16.360546,1.0,1.00,4.0,12.00,142.0


In [16]:
df.isnull().sum()*1000

,0
Index,0
Highest Charting Position,0
Number of Times Charted,0
Week of Highest Charting,0
Song Name,0
Streams,0
Artist,0
Artist Followers,0
Song ID,0
Genre,0


In [17]:
row_nulls_count = int(df.isnull().sum().sum())
row_nulls_count

0

In [18]:
print(f"Rows containing nulls: {row_nulls_count}")

Rows containing nulls: 0


No nulls! Great!

Now we have a look at the columns themselves

In [20]:
# nunique
df.nunique().sort_values( ascending = False )

,0
Index,1556
Streams,1556
Song Name,1556
Song ID,1517
Duration (ms),1486
Tempo,1461
Loudness,1394
Acousticness,965
Weeks Charted,775
Speechiness,772


In [21]:
# normalize to look for ID column(s)
df.nunique().sort_values( ascending = False ) / df.shape[0] * 100

,0
Index,100.000000
Streams,100.000000
Song Name,100.000000
Song ID,97.493573
Duration (ms),95.501285
Tempo,93.894602
Loudness,89.588689
Acousticness,62.017995
Weeks Charted,49.807198
Speechiness,49.614396


In [24]:
df['Song ID'].unique()

array(['3Wrjm47oTz2sjIgck11l5e', '5HCyWlXZPP0y6Gqq8TgA20',
       '4ZtFanR9U6ndgddUvNcjcG', ..., '1rfofaqEpACxVEHIZBJe6W',
       '5F8ffc8KWKNawllr5WsW0r', '3i9UVldZOE0aD0JnyfAZZ0'],
      shape=(1517,), dtype=object)

In [32]:
# Get a sense of the data
df.head().iloc[:1].transpose()

,0
Index,1
Highest Charting Position,1
Number of Times Charted,8
Week of Highest Charting,2021-07-23--2021-07-30
Song Name,Beggin'
Streams,"48,633,449"
Artist,Måneskin
Artist Followers,3377762
Song ID,3Wrjm47oTz2sjIgck11l5e
Genre,"['indie rock italiano', 'italian pop']"


We want to keep numeric columns while dropping difficult object columns, complicated strings, future target-correlated columns, and id columns.

In [33]:
df_bak1 = df.copy()

In [34]:
df = df_bak1.copy()

In [36]:
df.head()

,Index,Highest Charting Position,Number of Times Charted,Week of Highest Charting,Song Name,Streams,Artist,Artist Followers,Song ID,Genre,...,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence,Chord
0,1,1,8,2021-07-23--2021-07-30,Beggin',"48,633,449",Måneskin,3377762,3Wrjm47oTz2sjIgck11l5e,"['indie rock italiano', 'italian pop']",...,0.714,0.8,-4.808,0.0504,0.127,0.359,134.002,211560,0.589,B
1,2,2,3,2021-07-23--2021-07-30,STAY (with Justin Bieber),"47,248,719",The Kid LAROI,2230022,5HCyWlXZPP0y6Gqq8TgA20,['australian hip hop'],...,0.591,0.764,-5.484,0.0483,0.0383,0.103,169.928,141806,0.478,C#/Db
2,3,1,11,2021-06-25--2021-07-02,good 4 u,"40,162,559",Olivia Rodrigo,6266514,4ZtFanR9U6ndgddUvNcjcG,['pop'],...,0.563,0.664,-5.044,0.154,0.335,0.0849,166.928,178147,0.688,A
3,4,3,5,2021-07-02--2021-07-09,Bad Habits,"37,799,456",Ed Sheeran,83293380,6PQ88X9TkUIAUIZJHW2upE,"['pop', 'uk pop']",...,0.808,0.897,-3.712,0.0348,0.0469,0.364,126.026,231041,0.591,B
4,5,5,1,2021-07-23--2021-07-30,INDUSTRY BABY (feat. Jack Harlow),"33,948,454",Lil Nas X,5473565,27NovPIUIRrOZoCHxABJwK,"['lgbtq+ hip hop', 'pop rap']",...,0.736,0.704,-7.409,0.0615,0.0203,0.0501,149.995,212000,0.894,D#/Eb


In [37]:
ids = ['Index', 'Song Name', 'Song ID']
ids

['Index', 'Song Name', 'Song ID']

In [38]:
df = df.drop(columns = ids)

In [39]:
df.head().iloc[:1].transpose()

,0
Highest Charting Position,1
Number of Times Charted,8
Week of Highest Charting,2021-07-23--2021-07-30
Streams,"48,633,449"
Artist,Måneskin
Artist Followers,3377762
Genre,"['indie rock italiano', 'italian pop']"
Release Date,2017-12-08
Weeks Charted,2021-07-23--2021-07-30\n2021-07-16--2021-07-23...
Popularity,100


In [40]:
df_bak2 = df.copy()

In [46]:
df = df_bak2.copy()

In [42]:
df.head()

,Highest Charting Position,Number of Times Charted,Week of Highest Charting,Streams,Artist,Artist Followers,Genre,Release Date,Weeks Charted,Popularity,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence,Chord
0,1,8,2021-07-23--2021-07-30,"48,633,449",Måneskin,3377762,"['indie rock italiano', 'italian pop']",2017-12-08,2021-07-23--2021-07-30\n2021-07-16--2021-07-23...,100,0.714,0.8,-4.808,0.0504,0.127,0.359,134.002,211560,0.589,B
1,2,3,2021-07-23--2021-07-30,"47,248,719",The Kid LAROI,2230022,['australian hip hop'],2021-07-09,2021-07-23--2021-07-30\n2021-07-16--2021-07-23...,99,0.591,0.764,-5.484,0.0483,0.0383,0.103,169.928,141806,0.478,C#/Db
2,1,11,2021-06-25--2021-07-02,"40,162,559",Olivia Rodrigo,6266514,['pop'],2021-05-21,2021-07-23--2021-07-30\n2021-07-16--2021-07-23...,99,0.563,0.664,-5.044,0.154,0.335,0.0849,166.928,178147,0.688,A
3,3,5,2021-07-02--2021-07-09,"37,799,456",Ed Sheeran,83293380,"['pop', 'uk pop']",2021-06-25,2021-07-23--2021-07-30\n2021-07-16--2021-07-23...,98,0.808,0.897,-3.712,0.0348,0.0469,0.364,126.026,231041,0.591,B
4,5,1,2021-07-23--2021-07-30,"33,948,454",Lil Nas X,5473565,"['lgbtq+ hip hop', 'pop rap']",2021-07-23,2021-07-23--2021-07-30,96,0.736,0.704,-7.409,0.0615,0.0203,0.0501,149.995,212000,0.894,D#/Eb


In [47]:
future_features = [
    'Highest Charting Position',
    'Number of Times Charted',
    'Week of Highest Charting',
    'Weeks Charted'
]

In [48]:
df = df.drop(columns = future_features)

In [49]:
df.head().iloc[:1].transpose()

,0
Streams,"48,633,449"
Artist,Måneskin
Artist Followers,3377762
Genre,"['indie rock italiano', 'italian pop']"
Release Date,2017-12-08
Popularity,100
Danceability,0.714
Energy,0.8
Loudness,-4.808
Speechiness,0.0504


In [50]:
df_bak3 = df.copy()

In [51]:
df = df_bak3.copy()

In [52]:
df.head()

,Streams,Artist,Artist Followers,Genre,Release Date,Popularity,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence,Chord
0,"48,633,449",Måneskin,3377762,"['indie rock italiano', 'italian pop']",2017-12-08,100,0.714,0.8,-4.808,0.0504,0.127,0.359,134.002,211560,0.589,B
1,"47,248,719",The Kid LAROI,2230022,['australian hip hop'],2021-07-09,99,0.591,0.764,-5.484,0.0483,0.0383,0.103,169.928,141806,0.478,C#/Db
2,"40,162,559",Olivia Rodrigo,6266514,['pop'],2021-05-21,99,0.563,0.664,-5.044,0.154,0.335,0.0849,166.928,178147,0.688,A
3,"37,799,456",Ed Sheeran,83293380,"['pop', 'uk pop']",2021-06-25,98,0.808,0.897,-3.712,0.0348,0.0469,0.364,126.026,231041,0.591,B
4,"33,948,454",Lil Nas X,5473565,"['lgbtq+ hip hop', 'pop rap']",2021-07-23,96,0.736,0.704,-7.409,0.0615,0.0203,0.0501,149.995,212000,0.894,D#/Eb


In [53]:
target_proxies = ['Streams']
target_proxies

['Streams']

In [54]:
df = df.drop(columns = target_proxies)

In [55]:
df.head().iloc[:1].transpose()

,0
Artist,Måneskin
Artist Followers,3377762
Genre,"['indie rock italiano', 'italian pop']"
Release Date,2017-12-08
Popularity,100
Danceability,0.714
Energy,0.8
Loudness,-4.808
Speechiness,0.0504
Acousticness,0.127


In [56]:
df_bak4 = df.copy()

In [57]:
df = df_bak4.copy()

In [58]:
df.head()

,Artist,Artist Followers,Genre,Release Date,Popularity,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence,Chord
0,Måneskin,3377762,"['indie rock italiano', 'italian pop']",2017-12-08,100,0.714,0.8,-4.808,0.0504,0.127,0.359,134.002,211560,0.589,B
1,The Kid LAROI,2230022,['australian hip hop'],2021-07-09,99,0.591,0.764,-5.484,0.0483,0.0383,0.103,169.928,141806,0.478,C#/Db
2,Olivia Rodrigo,6266514,['pop'],2021-05-21,99,0.563,0.664,-5.044,0.154,0.335,0.0849,166.928,178147,0.688,A
3,Ed Sheeran,83293380,"['pop', 'uk pop']",2021-06-25,98,0.808,0.897,-3.712,0.0348,0.0469,0.364,126.026,231041,0.591,B
4,Lil Nas X,5473565,"['lgbtq+ hip hop', 'pop rap']",2021-07-23,96,0.736,0.704,-7.409,0.0615,0.0203,0.0501,149.995,212000,0.894,D#/Eb


In [59]:
categoricals = [
    'Artist',
    'Genre',
    'Chord',
    'Release Date'
]

In [60]:
df_categorical = df[categoricals]
df_categorical.head()

,Artist,Genre,Chord,Release Date
0,Måneskin,"['indie rock italiano', 'italian pop']",B,2017-12-08
1,The Kid LAROI,['australian hip hop'],C#/Db,2021-07-09
2,Olivia Rodrigo,['pop'],A,2021-05-21
3,Ed Sheeran,"['pop', 'uk pop']",B,2021-06-25
4,Lil Nas X,"['lgbtq+ hip hop', 'pop rap']",D#/Eb,2021-07-23


In [61]:
df = df.drop(columns = categoricals)

In [62]:
df.head().iloc[:1].transpose()

,0
Artist Followers,3377762
Popularity,100
Danceability,0.714
Energy,0.8
Loudness,-4.808
Speechiness,0.0504
Acousticness,0.127
Liveness,0.359
Tempo,134.002
Duration (ms),211560


In [63]:
df_bak5 = df.copy()

In [64]:
df = df_bak5.copy()

In [65]:
df.head()

,Artist Followers,Popularity,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence
0,3377762,100,0.714,0.8,-4.808,0.0504,0.127,0.359,134.002,211560,0.589
1,2230022,99,0.591,0.764,-5.484,0.0483,0.0383,0.103,169.928,141806,0.478
2,6266514,99,0.563,0.664,-5.044,0.154,0.335,0.0849,166.928,178147,0.688
3,83293380,98,0.808,0.897,-3.712,0.0348,0.0469,0.364,126.026,231041,0.591
4,5473565,96,0.736,0.704,-7.409,0.0615,0.0203,0.0501,149.995,212000,0.894


In [ ]:
# Want to keep numeric columns

In [66]:
df = df.apply(pd.to_numeric, errors='coerce')

In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1556 entries, 0 to 1555
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Artist Followers  1545 non-null   float64
 1   Popularity        1545 non-null   float64
 2   Danceability      1545 non-null   float64
 3   Energy            1545 non-null   float64
 4   Loudness          1545 non-null   float64
 5   Speechiness       1545 non-null   float64
 6   Acousticness      1545 non-null   float64
 7   Liveness          1545 non-null   float64
 8   Tempo             1545 non-null   float64
 9   Duration (ms)     1545 non-null   float64
 10  Valence           1545 non-null   float64
dtypes: float64(11)
memory usage: 133.8 KB


In [69]:
df.isnull().sum()

,0
Artist Followers,11
Popularity,11
Danceability,11
Energy,11
Loudness,11
Speechiness,11
Acousticness,11
Liveness,11
Tempo,11
Duration (ms),11


In [70]:
df[df['Popularity'].isna()].head()

,Artist Followers,Popularity,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence
35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
163,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
464,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
530,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
636,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [71]:
df = df.dropna()

In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1545 entries, 0 to 1555
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Artist Followers  1545 non-null   float64
 1   Popularity        1545 non-null   float64
 2   Danceability      1545 non-null   float64
 3   Energy            1545 non-null   float64
 4   Loudness          1545 non-null   float64
 5   Speechiness       1545 non-null   float64
 6   Acousticness      1545 non-null   float64
 7   Liveness          1545 non-null   float64
 8   Tempo             1545 non-null   float64
 9   Duration (ms)     1545 non-null   float64
 10  Valence           1545 non-null   float64
dtypes: float64(11)
memory usage: 144.8 KB
